In [3]:
import os
!pip install "numpy<2" "opencv-python" --force-reinstall
import cv2
import numpy as np
# Extensoes de imagem que o programa aceita. 
EXTENSOES = (".jpg", ".jpeg", ".png", ".bmp")

# ====================================================================== ===== # ETAPAS DE PROCESSAMENTO (cada funcao faz um passo) # ====================================================================== =====

def _preparar(gray):
  """Passo 1: limpa a imagem e tira a diferenca de iluminacao.
  - bilateralFilter: borra um pouco para tirar ruido, MAS mantem as bordas;
  - divisao pelo fundo borrado: deixa a imagem com brilho parecido em todo   lugar (tira sombra/canto escuro);
  - CLAHE: melhora o contraste local olhando o histograma de cada pedaco.     """
  suave = cv2.bilateralFilter(gray, d=7, sigmaColor=35, sigmaSpace=35)
  fundo = cv2.GaussianBlur(suave, (0, 0), sigmaX=25)   # so a iluminacao
  norm = cv2.divide(suave, fundo, scale=255)           # imagem / iluminacao
  clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
  return clahe.apply(norm)

def _energia_texto(realce, k=13):
    """Passo 2: cria um mapa que fica BRANCO onde tem tinta/letra.
    tophat  = realca coisas CLARAS pequenas (texto claro no fundo escuro);
    blackhat = realca coisas ESCURAS pequenas (texto escuro no fundo claro).
    Somando os dois, eu pego a tinta dos DOIS tipos de rotulo de uma vez.
    """
    elem = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    tophat = cv2.morphologyEx(realce, cv2.MORPH_TOPHAT, elem)
    blackhat = cv2.morphologyEx(realce, cv2.MORPH_BLACKHAT, elem)
    return cv2.add(tophat, blackhat)

def _binariza_tinta(energia, percentil=96, minimo=22):
    """Passo 3: transforma o mapa de tinta em preto-e-branco (0 ou 255).
    Uso um limiar ALTO (percentil 96): so a tinta bem forte (a letra do
    rotulo) vira branco. O vinco do plastico tem energia media e fica de fora.
    """
    energia = energia.copy()
    # O video tem um texto "FPS: 30.xx" no canto de cima/esquerda. Como ele
    # nao e produto, apago essa parte para nao virar um recorte errado.
    energia[0:55, 0:255] = 0
    thr = max(np.percentile(energia, percentil), minimo)
    _, binimg = cv2.threshold(energia, thr, 255, cv2.THRESH_BINARY)
    return binimg

def _blocos_de_texto(tinta):
    """Passo 4: junta as letrinhas soltas em BLOCOS (o rotulo inteiro).
    - OPEN: apaga pontinhos isolados (sujeira/ruido);
    - CLOSE: gruda letras e palavras vizinhas formando um retangulo cheio;
    - dilate: engorda um pouco para fechar buracos.
    """
    binimg = cv2.morphologyEx(
        tinta, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
    )
    binimg = cv2.morphologyEx(
        binimg, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (29, 21)),
    )
    binimg = cv2.dilate(
        binimg, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    )
    return binimg

def _unir_caixas(caixas, largura, altura):
    """Junta caixas que se sobrepoem, para nao salvar o mesmo rotulo 2 vezes.
    Vai comparando as caixas de duas em duas: se elas se cruzam bastante,
    viram uma caixa so (a que cobre as duas). Repete ate nao mudar mais nada.
    """
    caixas = list(caixas)
    mudou = True
    while mudou:
        mudou = False
        novas = []
        usados = [False] * len(caixas)
        for i in range(len(caixas)):
            if usados[i]:
                continue
            ax1, ay1, ax2, ay2 = caixas[i]
            for j in range(i + 1, len(caixas)):
                if usados[j]:
                    continue
                bx1, by1, bx2, by2 = caixas[j]
                # area onde as duas caixas se cruzam
                ix1, iy1 = max(ax1, bx1), max(ay1, by1)
                ix2, iy2 = min(ax2, bx2), min(ay2, by2)
                if ix2 > ix1 and iy2 > iy1:
                    inter = (ix2 - ix1) * (iy2 - iy1)
                    menor = min((ax2 - ax1) * (ay2 - ay1),
                                (bx2 - bx1) * (by2 - by1))
                    if inter > 0.20 * menor:     # cruzam mais de 20% -> juntar
                        ax1, ay1 = min(ax1, bx1), min(ay1, by1)
                        ax2, ay2 = max(ax2, bx2), max(ay2, by2)
                        usados[j] = True
                        mudou = True
            usados[i] = True
            novas.append((ax1, ay1, ax2, ay2))
        caixas = novas
    return caixas

# Os numeros (tamanho minimo, etc.) foram ajustados em imagens de 1280 de
# largura. Para funcionar igual em imagens de outro tamanho, eu redimensiono
# tudo para essa largura antes de processar.
LARGURA_PADRAO = 1280

# ===========================================================================
# FUNCAO PRINCIPAL: acha os rotulos de UMA imagem
# ===========================================================================
def segmentar_imagem(img_bgr):
    """Recebe a imagem e devolve a lista de caixas (x1, y1, x2, y2) dos
    rotulos, ja em coordenadas da imagem original e ja com uma folga para
    incluir o nome do produto."""

    # --- deixa a imagem com 1280 de largura (se nao for) ---
    h0, w0 = img_bgr.shape[:2]
    if abs(w0 - LARGURA_PADRAO) > 8:
        nh = max(1, round(h0 * LARGURA_PADRAO / w0))
        proc = cv2.resize(img_bgr, (LARGURA_PADRAO, nh))
    else:
        proc = img_bgr

    inv = w0 / proc.shape[1]  # usado no fim para voltar ao tamanho original

    # --- passos 1 a 4 (ver as funcoes acima) ---
    gray = cv2.cvtColor(proc, cv2.COLOR_BGR2GRAY)  # tira a cor (ja e cinza)
    altura, largura = gray.shape
    area_img = altura * largura

    realce = _preparar(gray) # Certifique-se de que a função _preparar() está definida no seu arquivo
    energia = _energia_texto(realce)
    tinta = _binariza_tinta(energia)  # preto-e-branco da tinta
    blocos = _blocos_de_texto(tinta)  # blocos (candidatos a rotulo)

    # --- passo 5: separa cada bloco branco (componentes conexos) ---
    num, _, stats, _ = cv2.connectedComponentsWithStats(blocos, 8)
    caixas = []

    for i in range(1, num):  # i=0 e o fundo, pulo
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area_caixa = w * h

        # algumas medidas do bloco para decidir se ele e mesmo um rotulo
        proporcao = w / max(h, 1)  # largura / altura
        sub_tinta = tinta[y:y + h, x:x + w]
        dens_tinta = cv2.countNonZero(sub_tinta) / max(area_caixa, 1)  # % tinta
        n_tracos = cv2.connectedComponents(sub_tinta, 8)[0] - 1  # qtd de letras
        preench = stats[i, cv2.CC_STAT_AREA] / max(area_caixa, 1)  # quao cheio
        sub = gray[y:y + h, x:x + w]
        brilho = sub.mean()  # media de cinza
        contraste = sub.std()  # variacao de cinza
        area_pct = area_caixa / area_img  # tamanho relativo

        # -------- FILTROS: jogo fora tudo que NAO parece um rotulo --------
        # (a regra do trabalho diz que recorte errado = falso positivo = erro,
        #  entao prefiro ser rigoroso e errar pouco.)
        if area_pct < 0.004:  # bloco pequeno demais
            continue
        if area_pct > 0.22:  # bloco grande demais (fundo)
            continue
        if w < 44 or h < 32:  # estreito/baixo demais
            continue
        if w > 0.70 * largura or h > 0.70 * altura:
            continue
        if proporcao < 0.25 or proporcao > 4.2:  # fita muito fina ou coluna
            continue
        if dens_tinta < 0.23:  # pouca tinta (plastico, logo SS)
            continue
        if dens_tinta > 0.75:  # tinta demais (codigo de barras)
            continue
        if preench < 0.42:  # bloco "esfarrapado"
            continue
        if brilho < 80:  # muito escuro (cesta/sombra)
            continue
        if contraste < 15:  # liso, sem texto
            continue

        # bloco grande, com poucas letras e tinta so media = e um simbolo/logo
        # (ex.: o logo "SS" da caixa de papelao), nao o texto do produto.
        if area_pct > 0.020 and n_tracos <= 4 and dens_tinta < 0.28:
            continue

        # -------- aumenta um pouco a caixa para pegar o NOME do produto --------
        # O nome fica colado na marca "SUPER FRANGO". Pegar um pouco a mais nao
        # e penalizado; salvar SEM o nome (so "frango") seria erro. As caixas
        # que crescerem e se encostarem sao unidas depois.
        pad = int(min(max(0.80 * max(w, h), 45), 180))
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(largura, x + w + pad)
        y2 = min(altura, y + h + pad)
        caixas.append((x1, y1, x2, y2))

    # junta caixas sobrepostas
    caixas = _unir_caixas(caixas, largura, altura)

    # se eu tinha redimensionado, volto as caixas para o tamanho original
    if inv != 1.0:
        caixas = [(min(int(round(x1 * inv)), w0), min(int(round(y1 * inv)), h0),
                   min(int(round(x2 * inv)), w0), min(int(round(y2 * inv)), h0))
                  for (x1, y1, x2, y2) in caixas]

    return caixas

# ===========================================================================
# PERCORRER O DATASET E SALVAR OS RECORTES
# ===========================================================================
def processar_dataset(pasta_dataset="dataset", pasta_saida="resultado",
                      indices=None):
    """Passa por todas as pastas e imagens e salva os recortes em
    resultado/<classe>/<nome>_segmentada_N.png.

    indices: None (padrao) = processa TODAS as imagens.
             lista (ex.: [0, 16, 33, 49]) = processa so essas posicoes de cada
             pasta (serve para gerar uma amostra rapida e conferir antes)."""

    if not os.path.isdir(pasta_dataset):
        print(f"ERRO: pasta '{pasta_dataset}' nao encontrada.")
        return

    total_imgs = 0
    total_recortes = 0

    # para cada pasta de produto (classe)
    for classe in sorted(os.listdir(pasta_dataset)):
        dir_classe = os.path.join(pasta_dataset, classe)
        if not os.path.isdir(dir_classe):
            continue

        # lista as imagens da pasta
        # Obs: A constante 'EXTENSOES' precisa estar declarada (ex: EXTENSOES = ('.png', '.jpg'))
        arqs = [a for a in sorted(os.listdir(dir_classe))
                if a.lower().endswith(EXTENSOES)]

        if indices is not None:
            # modo amostra
            arqs = [arqs[i] for i in indices if i < len(arqs)]

        # cria a pasta de saida correspondente
        dir_saida = os.path.join(pasta_saida, classe)
        os.makedirs(dir_saida, exist_ok=True)

        # para cada imagem, acha os rotulos e salva os recortes
        for arq in arqs:
            caminho = os.path.join(dir_classe, arq)
            img = cv2.imread(caminho)

            if img is None:
                # arquivo invalido: pula
                continue

            total_imgs += 1
            caixas = segmentar_imagem(img)
            base = os.path.splitext(arq)[0]  # nome sem a extensao

            for n, (x1, y1, x2, y2) in enumerate(caixas, start=1):
                recorte = img[y1:y2, x1:x2]  # corta o pedaco da imagem
                destino = os.path.join(dir_saida, f"{base}segmentada{n}.png")
                cv2.imwrite(destino, recorte)  # salva o recorte
                total_recortes += 1

    print(f"Concluido: {total_imgs} imagens -> {total_recortes} recortes "
          f"em '{pasta_saida}/'.")

# Quando rodar "python solucao.py", processa o dataset inteiro.
#if _name_ == "_main_":
processar_dataset("./work/Train_and_Validation", "resultado")

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached opencv_python-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Using cached opencv_python-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (63.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled